# T4 - Sequential Runtime and VRAM Safety

**Guardian Eye task:** T4

Run this notebook top-to-bottom on the target virtual machine. Configuration is centralized near the top, and heavy model work only occurs in explicitly marked execution cells.


## 1. Runtime utilities

Provides logging, CUDA-memory snapshots, and aggressive cleanup between heavy stages.


In [ ]:
from __future__ import annotations
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Any, Callable, Iterator
import gc
import logging
import time

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("guardian_eye.runtime")

def cuda_snapshot(label: str) -> dict[str, float | str]:
    try:
        import torch
        if not torch.cuda.is_available():
            return {"label": label, "device": "cpu"}
        gib = 1024 ** 3
        return {
            "label": label,
            "device": torch.cuda.get_device_name(0),
            "allocated_gib": round(torch.cuda.memory_allocated() / gib, 3),
            "reserved_gib": round(torch.cuda.memory_reserved() / gib, 3),
            "free_gib": round(torch.cuda.mem_get_info()[0] / gib, 3),
        }
    except ImportError:
        return {"label": label, "device": "torch-not-installed"}

def free_runtime_memory() -> None:
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except ImportError:
        pass


## 2. Load-on-demand model lease

Ensures only one heavy model is leased at a time and always calls its optional `unload()` method.


In [ ]:
@contextmanager
def model_lease(name: str, loader: Callable[[], Any]) -> Iterator[Any]:
    logger.info("Loading %s | %s", name, cuda_snapshot("before_load"))
    model = None
    started = time.perf_counter()
    try:
        model = loader()
        logger.info("Loaded %s in %.2fs | %s", name, time.perf_counter() - started, cuda_snapshot("after_load"))
        yield model
    finally:
        if model is not None:
            unload = getattr(model, "unload", None)
            if callable(unload):
                unload()
            del model
        free_runtime_memory()
        logger.info("Unloaded %s | %s", name, cuda_snapshot("after_unload"))


## 3. Strict sequential runner

The classifier is fully released before the VLM is loaded. The text LLM is a separate optional stage and is never resident with either model.


In [ ]:
@dataclass
class RuntimeLoaders:
    classifier: Callable[[], Any]
    vlm: Callable[[], Any]
    text_llm: Callable[[], Any] | None = None

class SequentialRuntime:
    def __init__(self, loaders: RuntimeLoaders):
        self.loaders = loaders

    def analyze_clip(self, video_path: str, packet_fn, retrieve_fn, persist_fn, recap_fn):
        with model_lease("V9 RLVS classifier", self.loaders.classifier) as classifier:
            core = classifier.analyze_clip(video_path)

        # CPU-only/RAG stages run after classifier VRAM is released.
        packet = packet_fn(core.telemetry, core.gate, core.gqs)
        refs = retrieve_fn(packet["packet_summary"], top_k=4)

        with model_lease("Qwen2.5-VL narrator", self.loaders.vlm) as vlm:
            narrative = vlm.generate_clip_explanation(
                frames=core.frames_vit,
                packet=packet,
                refs=refs,
                verdict=core.verdict,
                confidence=core.confidence,
            )

        if narrative.get("verdict", core.verdict) != core.verdict:
            raise RuntimeError("Narrator contradicted the classifier verdict")

        recap_paths = recap_fn(core.frames_vit, packet)
        incident = persist_fn(core=core, packet=packet, narrative=narrative, frames_ref=recap_paths)
        return core, packet, narrative, incident, recap_paths

    def summarize_history(self, records, question: str):
        if self.loaders.text_llm is None:
            raise RuntimeError("No text LLM loader configured")
        with model_lease("historical-query text LLM", self.loaders.text_llm) as llm:
            return llm.summarize_incidents(records, question)


## 4. 12 GB GPU preflight and fallback policy

Checks available VRAM before a heavy load and raises a clear error instead of risking an out-of-memory crash.


In [ ]:
def require_free_vram(minimum_gib: float, stage: str) -> None:
    snapshot = cuda_snapshot(stage)
    free = snapshot.get("free_gib")
    if isinstance(free, float) and free < minimum_gib:
        raise RuntimeError(
            f"{stage} requires at least {minimum_gib:.1f} GiB free VRAM; "
            f"only {free:.1f} GiB is available. Use a smaller quantized model or CPU offload."
        )

FALLBACK_POLICY = {
    "classifier": "Run first, then unload completely.",
    "vlm": "Use 4-bit AWQ, device_map='auto', and CPU offload if needed.",
    "text_llm": "Load only for historical summaries; prefer a smaller quantized local model.",
    "embeddings": "Keep on CPU for the 12 GB demo machine.",
}
FALLBACK_POLICY


## 5. Runtime logging acceptance check

Use these assertions after a target-VM demo run to verify that each stage emitted load/unload boundaries.


In [ ]:
REQUIRED_LOG_MARKERS = [
    "Loading V9 RLVS classifier",
    "Unloaded V9 RLVS classifier",
    "Loading Qwen2.5-VL narrator",
    "Unloaded Qwen2.5-VL narrator",
]

def assert_sequential_log(log_text: str) -> None:
    positions = [log_text.index(marker) for marker in REQUIRED_LOG_MARKERS]
    assert positions == sorted(positions), "Heavy-model stages did not execute sequentially"
    print("Sequential load/unload order verified.")
